In [10]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from collections import defaultdict
from tqdm import tqdm

from sklearn.metrics import silhouette_score

from time_series.data_generators import LorenzGenerator
from time_series.models import KernelRidgeRegression, RascuttiModel
from time_series.evaluators import MeanSquaredError

from experiment_logging import Experiment
from time_series_clustering import TimeSeriesClustering

# MSE vs noise

## Kernel Ridge Regression

In [7]:
experiment = Experiment("MSE vs Noise - KRR", "experiments")

In [ ]:
n_per = 10
n_groups = 2
seed = 0

np.random.seed(seed)

experiment.add_config(n_per=n_per, n_groups=n_groups, seed=seed, sweep="noise")

In [ ]:
noise_range = tqdm(np.linspace(1e-3, 10, 20))

for noise in noise_range:
    noise_range.set_description(f"Generating data for noise = {noise}")
    # Generate data groups
    datasets = []
    labels = []
    for i in range(n_groups):
        # Set group parameters
        rho = np.random.random()*30*2
        sigma = np.random.random()*10*2
        beta = np.random.random()*2*2

        for n in range(n_per):
            x0 = np.random.random(size=3)*10

            generator = LorenzGenerator(
                noise_covariance=noise,
                x0=x0,
                dt=0.01,
                T=10,
                rho=rho,
                sigma=sigma,
                beta=beta
            )

            t, data = generator()

            experiment.save_dataset(
                data,
                name=f"lorenz-group_{i}-id_{n}",
                metadata=dict(
                    group = i,
                    sample_id=n,
                    data_gen_info=dict(
                        x0 = x0.tolist(),
                        rho=float(rho),
                        sigma=float(sigma),
                        beta=float(beta),
                        noise=float(noise)
                    )
                )
            )

            datasets.append(
                data
            )

            labels.append(i)

    noise_range.set_description(f"Clustering for noise = {noise}")

    clustering_model = TimeSeriesClustering(
        model_cls=KernelRidgeRegression,
        kernel="rbf",
        split=[0.5, 0.4, 0.1],
        n_trials=50,
        n_jobs=-1,
    )

    clustering_model.fit(datasets)
    similarity_matrix = clustering_model.get_similarity_matrix()
    error_matrix = clustering_model.error_matrix()
    score = silhouette_score(similarity_matrix, labels=labels, metric="precomputed")

    experiment.add_result(
        results=dict(
            noise=noise,
            silhouette=score,
            similarity_matrix=similarity_matrix.tolist(),
            error_matrix=error_matrix.tolist()
        )
    )
